In [1]:
# ============================================================
# Build daily GPP and partition annual GPP into ECUP / LCUP
# based on newly calculated CUP and GPPmax files.
#
# ECUP = CUPstart to POS/GPPmax
# LCUP = POS/GPPmax to CUPend
#
# POS day GPP is split half to ECUP and half to LCUP.
# ============================================================

import os
import numpy as np
import pandas as pd


# ============================================================
# 1. CONFIG
# ============================================================
SIM_DIR = "../../data_results/1_data_source/1_daily_simulations"
PHENO_DIR = "../../data_results/2_results/2-3_gppmax_cup_6methods"
OUT_DIR = "../../data_results/2_results/2-6_gpp_pos_partition"
os.makedirs(OUT_DIR, exist_ok=True)

PHENO_FILE_TEMPLATE = (
    "calculated_GPPmax_CUP_ampFracs_10_20_30_25_thresholdScope_{species}.xlsx"
)

PLOTS = ["P04", "P06", "P08", "P10", "P11", "P13", "P16", "P17", "P19", "P20"]

SPECIES_LIST = ["ecosystem", "tree", "shrub", "sphagnum"]

PLOT_INFO = {
    "P06": {"co2": 0,   "warming": 0.00},
    "P20": {"co2": 0,   "warming": 2.25},
    "P13": {"co2": 0,   "warming": 4.50},
    "P08": {"co2": 0,   "warming": 6.75},
    "P17": {"co2": 0,   "warming": 9.00},

    "P19": {"co2": 500, "warming": 0.00},
    "P11": {"co2": 500, "warming": 2.25},
    "P04": {"co2": 500, "warming": 4.50},
    "P16": {"co2": 500, "warming": 6.75},
    "P10": {"co2": 500, "warming": 9.00},
}

BASE_YEARS = [2011, 2012, 2013]
ANOM_YEARS = list(range(2014, 2022))

START_YEAR = 2011
END_YEAR = 2021

# ---- columns from new CUP/GPPmax file ----
POS_COLS = [
    "doy_gppmax_weibull",
    "doy_gppmax_hants",
    "doy_gppmax_savgol",
]

CUP_START_COL = "cup_start_whsM_25"
CUP_END_COL   = "cup_end_whsM_25"


# ============================================================
# 2. Master no-leap calendar
# ============================================================
def build_no_leap_calendar(start_year=2011, end_year=2021):
    dates = pd.date_range(
        start=f"{start_year}-01-01",
        end=f"{end_year}-12-31",
        freq="D"
    )

    df = pd.DataFrame({"date": dates})
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day

    df = df[~((df["month"] == 2) & (df["day"] == 29))].copy()
    df["doy"] = df.groupby("year").cumcount() + 1

    return df[["date", "year", "doy"]].reset_index(drop=True)


DF_CALENDAR = build_no_leap_calendar(START_YEAR, END_YEAR)


# ============================================================
# 3. Daily GPP from simulation
# ============================================================
def calc_species_gpp(df, species):
    if species == "ecosystem":
        return (
            0.25 * df["gpp_Shrub"]
            + 0.25 * df["gpp_Sphagnum"]
            + 0.50 * df["gpp_Tree"]
        ) * 24.0

    if species == "tree":
        return 0.50 * df["gpp_Tree"] * 24.0

    if species == "shrub":
        return 0.25 * df["gpp_Shrub"] * 24.0

    if species == "sphagnum":
        return 0.25 * df["gpp_Sphagnum"] * 24.0

    raise ValueError(f"Unknown species: {species}")


def read_daily_gpp_one_plot(plot, species):
    f_csv = os.path.join(
        SIM_DIR,
        f"TECO-SPRUCE_run_mcmc_alltreat_{plot}_Daily.csv"
    )

    df = pd.read_csv(f_csv)

    df["datetime"] = (
        pd.to_datetime(df[" year"].astype(int).astype(str) + "-01-01")
        + pd.to_timedelta(df["doy"] - 1, unit="D")
    )

    df["GPP"] = calc_species_gpp(df, species)

    df = df[["datetime", "GPP"]].copy()
    df = df.set_index("datetime").loc[f"{START_YEAR}":f"{END_YEAR}"]

    daily = df.resample("D").mean().reset_index()
    daily = daily.rename(columns={"datetime": "date"})

    daily["date"] = pd.to_datetime(daily["date"])
    daily["year"] = daily["date"].dt.year
    daily["month"] = daily["date"].dt.month
    daily["day"] = daily["date"].dt.day

    daily = daily[~((daily["month"] == 2) & (daily["day"] == 29))].copy()

    daily = daily.merge(
        DF_CALENDAR,
        on=["date", "year"],
        how="right"
    )

    daily["plot"] = plot
    daily["GPP"] = daily["GPP"].fillna(0.0)

    daily["co2"] = PLOT_INFO[plot]["co2"]
    daily["warming"] = PLOT_INFO[plot]["warming"]

    return daily[["date", "year", "doy", "plot", "co2", "warming", "GPP"]]


def build_daily_gpp(species):
    all_daily = []

    for plot in PLOTS:
        daily = read_daily_gpp_one_plot(plot, species)
        all_daily.append(daily)

    df_daily = pd.concat(all_daily, ignore_index=True)
    return df_daily


# ============================================================
# 4. Read new CUP / GPPmax file
# ============================================================
def read_phenology_file(species):
    f_xlsx = os.path.join(
        PHENO_DIR,
        PHENO_FILE_TEMPLATE.format(species=species)
    )

    df = pd.read_excel(f_xlsx)

    needed_cols = [
        "plot", "year",
        "temp_mean", "temp_gs_mean", "temp_max",
        "gpp_sum",
        CUP_START_COL,
        CUP_END_COL,
    ] + POS_COLS

    missing = [c for c in needed_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{species}: missing columns: {missing}")

    df["doy_gppmax"] = df[POS_COLS].mean(axis=1)
    df["cup_start"] = df[CUP_START_COL]
    df["cup_end"] = df[CUP_END_COL]

    if "co2" not in df.columns:
        df["co2"] = df["plot"].map(lambda x: PLOT_INFO[x]["co2"])

    if "warming" not in df.columns:
        df["warming"] = df["plot"].map(lambda x: PLOT_INFO[x]["warming"])

    keep_cols = [
        "plot", "year", "co2", "warming",
        "temp_mean", "temp_gs_mean", "temp_max",
        "gpp_sum", "doy_gppmax", "cup_start", "cup_end"
    ]

    df = df[keep_cols].copy()

    df = (
        df.groupby(["plot", "year", "co2", "warming"], as_index=False)
        .agg({
            "temp_mean": "mean",
            "temp_gs_mean": "mean",
            "temp_max": "mean",
            "gpp_sum": "mean",
            "doy_gppmax": "mean",
            "cup_start": "mean",
            "cup_end": "mean",
        })
    )

    return df


# ============================================================
# 5. Partition GPP within CUP
# ============================================================
def partition_gpp_ecup_lcup(df_daily, df_pheno):
    out_rows = []

    for _, row in df_pheno.iterrows():
        plot = row["plot"]
        year = int(row["year"])

        pos = row["doy_gppmax"]
        cup_start = row["cup_start"]
        cup_end = row["cup_end"]

        if not np.isfinite(pos) or not np.isfinite(cup_start) or not np.isfinite(cup_end):
            continue

        pos_doy = int(np.round(pos))
        start_doy = int(np.round(cup_start))
        end_doy = int(np.round(cup_end))

        pos_doy = max(1, min(365, pos_doy))
        start_doy = max(1, min(365, start_doy))
        end_doy = max(1, min(365, end_doy))

        if start_doy > pos_doy:
            start_doy = pos_doy

        if end_doy < pos_doy:
            end_doy = pos_doy

        df_sub = df_daily[
            (df_daily["plot"] == plot)
            & (df_daily["year"] == year)
        ].copy()

        if df_sub.empty:
            continue

        df_ecup_before = df_sub[
            (df_sub["doy"] >= start_doy)
            & (df_sub["doy"] < pos_doy)
        ]

        df_lcup_after = df_sub[
            (df_sub["doy"] > pos_doy)
            & (df_sub["doy"] <= end_doy)
        ]

        df_peak = df_sub[df_sub["doy"] == pos_doy]

        gpp_peak = df_peak["GPP"].sum() if not df_peak.empty else 0.0

        gpp_ecup = df_ecup_before["GPP"].sum() + 0.5 * gpp_peak
        gpp_lcup = df_lcup_after["GPP"].sum() + 0.5 * gpp_peak

        ecup = float(pos_doy - start_doy + 1)
        lcup = float(end_doy - pos_doy + 1)
        cup = float(end_doy - start_doy + 1)

        gpp_rate_ecup = gpp_ecup / ecup if ecup > 0 else np.nan
        gpp_rate_lcup = gpp_lcup / lcup if lcup > 0 else np.nan
        gpp_rate_cup = (gpp_ecup + gpp_lcup) / cup if cup > 0 else np.nan

        out_rows.append({
            "plot": plot,
            "year": year,
            "co2": row["co2"],
            "warming": row["warming"],

            "temp_mean": row["temp_mean"],
            "temp_gs_mean": row["temp_gs_mean"],
            "temp_max": row["temp_max"],

            "gpp_sum": row["gpp_sum"],
            "doy_gppmax": row["doy_gppmax"],
            "cup_start": row["cup_start"],
            "cup_end": row["cup_end"],

            "pos_doy_used": pos_doy,
            "cup_start_used": start_doy,
            "cup_end_used": end_doy,

            "CUP": cup,
            "ECUP": ecup,
            "LCUP": lcup,

            "GPP_ecup": gpp_ecup,
            "GPP_lcup": gpp_lcup,
            "GPP_cup": gpp_ecup + gpp_lcup,

            "GPP_rate_ecup": gpp_rate_ecup,
            "GPP_rate_lcup": gpp_rate_lcup,
            "GPP_rate_cup": gpp_rate_cup,
        })

    return pd.DataFrame(out_rows)


# ============================================================
# 6. Add baseline and anomaly
# ============================================================
def add_baseline_anomaly(df):
    value_cols = [
        "temp_mean", "temp_gs_mean", "temp_max",
        "gpp_sum",
        "doy_gppmax", "cup_start", "cup_end",
        "CUP", "ECUP", "LCUP",
        "GPP_ecup", "GPP_lcup", "GPP_cup",
        "GPP_rate_ecup", "GPP_rate_lcup", "GPP_rate_cup",
    ]

    df_base = (
        df[df["year"].isin(BASE_YEARS)]
        .groupby("plot", as_index=False)[value_cols]
        .mean()
    )

    df_base = df_base.rename(columns={c: f"{c}_base" for c in value_cols})

    df_out = df.merge(df_base, on="plot", how="left")

    for c in value_cols:
        df_out[f"{c}_anom"] = df_out[c] - df_out[f"{c}_base"]

    mask = ~df_out["year"].isin(ANOM_YEARS)

    for c in value_cols:
        df_out.loc[mask, f"{c}_anom"] = np.nan

    return df_out


# ============================================================
# 7. Optional wide daily output
# ============================================================
def save_daily_outputs(df_daily, species):
    f_long = os.path.join(
        OUT_DIR,
        f"TECO-SPRUCE_DA_{species}_daily_long_2011-2021.xlsx"
    )
    df_daily.to_excel(f_long, index=False)

    df_wide = (
        df_daily
        .pivot_table(
            index=["date", "year", "doy"],
            columns="plot",
            values="GPP",
            aggfunc="mean"
        )
        .reset_index()
    )

    df_wide.columns.name = None

    f_wide = os.path.join(
        OUT_DIR,
        f"TECO-SPRUCE_DA_{species}_daily_wide_2011-2021.xlsx"
    )
    df_wide.to_excel(f_wide, index=False)


# ============================================================
# 8. Main
# ============================================================
def main():
    for species in SPECIES_LIST:
        print(f"\nProcessing: {species}")

        df_daily = build_daily_gpp(species)
        save_daily_outputs(df_daily, species)

        df_pheno = read_phenology_file(species)

        df_summary = partition_gpp_ecup_lcup(
            df_daily=df_daily,
            df_pheno=df_pheno
        )

        df_summary = add_baseline_anomaly(df_summary)

        df_summary = (
            df_summary
            .sort_values(["plot", "year"])
            .reset_index(drop=True)
        )

        out_file = os.path.join(
            OUT_DIR,
            f"GPP_POS_ECUP_LCUP_summary_{species}.xlsx"
        )

        df_summary.to_excel(out_file, index=False)

        print(f"Saved: {out_file}")


if __name__ == "__main__":
    main()


Processing: ecosystem
Saved: ../../data_results/2_results/2-6_gpp_pos_partition/GPP_POS_ECUP_LCUP_summary_ecosystem.xlsx

Processing: tree
Saved: ../../data_results/2_results/2-6_gpp_pos_partition/GPP_POS_ECUP_LCUP_summary_tree.xlsx

Processing: shrub
Saved: ../../data_results/2_results/2-6_gpp_pos_partition/GPP_POS_ECUP_LCUP_summary_shrub.xlsx

Processing: sphagnum
Saved: ../../data_results/2_results/2-6_gpp_pos_partition/GPP_POS_ECUP_LCUP_summary_sphagnum.xlsx
